In [1]:
import pandas as pd
import numpy as np


n_samples = 1000

np.random.seed(42)


gpa = np.random.normal(loc=2.7, scale=0.5, size=n_samples)
gpa = np.clip(gpa, 1.0, 4.0)


income_brackets = ['Low', 'Medium', 'High']
household_income = np.random.choice(income_brackets, size=n_samples, p=[0.5, 0.4, 0.1])


mental_health_score = np.random.normal(loc=5.5, scale=2.0, size=n_samples)
mental_health_score = np.clip(mental_health_score, 1.0, 10.0)


dropout_risk = []
for i in range(n_samples):
    risk_prob = 0.1
    if gpa[i] < 2.2:
        risk_prob += 0.4
    if household_income[i] == 'Low':
        risk_prob += 0.3
    if mental_health_score[i] > 7.0:
        risk_prob += 0.2


    status = 1 if np.random.rand() < risk_prob else 0
    dropout_risk.append(status)


data = pd.DataFrame({
    'Student_ID': [f'STU_{i:04d}' for i in range(1, n_samples + 1)],
    'Cumulative_GPA': np.round(gpa, 2),
    'Household_Income': household_income,
    'Mental_Health_Index': np.round(mental_health_score, 1),
    'Dropout_Status': dropout_risk
})


data.to_csv('sri_lankan_student_dropout_dataset.csv', index=False)
print("Dataset generated and saved successfully!")

Dataset generated and saved successfully!


In [2]:
!pip install imbalanced-learn

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE


df = pd.read_csv('sri_lankan_student_dropout_dataset.csv')


X = df.drop(columns=['Student_ID', 'Dropout_Status'])
y = df['Dropout_Status']

numeric_features = ['Cumulative_GPA', 'Mental_Health_Index']
categorical_features = ['Household_Income']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

X_processed = preprocessor.fit_transform(X)


smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_processed, y)

print("Preprocessing and SMOTE balancing completed!")

Preprocessing and SMOTE balancing completed!


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)


baseline_model = LogisticRegression(random_state=42)
baseline_model.fit(X_train, y_train)


y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]


f1 = f1_score(y_test, y_pred, average='macro')
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Macro-averaged F1-Score: {f1:.4f}")
print(f"AUC-ROC Score: {roc_auc:.4f}")

Macro-averaged F1-Score: 0.6860
AUC-ROC Score: 0.7153


In [5]:
!pip install fpdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 14.6 MB/s eta 0:00:00


In [6]:
from fpdf import FPDF

class PDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 12)
        self.cell(0, 10, 'IT41043 Intelligent Systems - Baseline Model Report', 0, 1, 'C')
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')


pdf = PDF()
pdf.add_page()
pdf.set_font('Arial', '', 11)


pdf.cell(0, 10, 'Student ID: ITBIN-2313-0033', 0, 1)
pdf.cell(0, 10, 'Module: IT41043 Intelligent Systems', 0, 1)
pdf.ln(5)

pdf.set_font('Arial', 'B', 12)
pdf.cell(0, 10, 'Baseline Model Results (Logistic Regression / TF-IDF):', 0, 1)

pdf.set_font('Arial', '', 11)

pdf.cell(0, 10, f'- Macro-averaged F1-Score: {f1:.4f}', 0, 1)
pdf.cell(0, 10, f'- ROC-AUC Score: {roc_auc}', 0, 1)


pdf_output_path = 'Baseline_Model_Report.pdf'
pdf.output(pdf_output_path)

from google.colab import files
files.download(pdf_output_path)

/tmp/ipykernel_1657/3749383592.py:5: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  self.set_font('Arial', 'B', 12)
/tmp/ipykernel_1657/3749383592.py:6: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 10, 'IT41043 Intelligent Systems - Baseline Model Report', 0, 1, 'C')
/tmp/ipykernel_1657/3749383592.py:17: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font('Arial', '', 11)
/tmp/ipykernel_1657/3749383592.py:20: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, 'Student ID: ITBIN-2313-0033', 0, 1)
/tmp/ipykernel_1657/3749383592.py:21: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGI

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>